In [1]:
import os
import shutil
import random
import yaml
from pathlib import Path

In [2]:
# Config
LABELING_DIR  = "../data/yolo_labeling"       # Ảnh + label .txt từ LabelImg
YOLO_DATASET  = "../data/yolo_dataset"         # Output dataset theo cấu trúc YOLO

CLASS_NAMES   = ["rice_leaf"]                  # Tên class dùng khi label
TRAIN_RATIO   = 0.8
VAL_RATIO     = 0.2

random.seed(42)

In [3]:
# ── 1. Tạo cấu trúc thư mục ───────────────────────────────────────────
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DATASET, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET, 'labels', split), exist_ok=True)

print(" Created YOLO dataset structure:")
print(f"   {YOLO_DATASET}/")
print(f"   ├── images/train/")
print(f"   ├── images/val/")
print(f"   ├── labels/train/")
print(f"   └── labels/val/")

 Created YOLO dataset structure:
   ../data/yolo_dataset/
   ├── images/train/
   ├── images/val/
   ├── labels/train/
   └── labels/val/


In [4]:
# 2. Thu thập các cặp image-label hợp lệ
all_images = [f for f in os.listdir(LABELING_DIR)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

valid_pairs = []
skipped = []

for img_file in all_images:
    stem = Path(img_file).stem
    label_file = stem + '.txt'
    label_path = os.path.join(LABELING_DIR, label_file)
    
    if not os.path.exists(label_path):
        skipped.append(img_file + ' (no label file)')
        continue
    
    # Bỏ qua label rỗng
    with open(label_path, 'r') as f:
        content = f.read().strip()
    if not content:
        skipped.append(img_file + ' (empty label)')
        continue
    
    valid_pairs.append((img_file, label_file))

print(f"Valid pairs : {len(valid_pairs)}")
print(f"Skipped     : {len(skipped)}")
if skipped:
    print("  Skipped files:")
    for s in skipped[:5]:
        print(f"    - {s}")

Valid pairs : 1197
Skipped     : 1092
  Skipped files:
    - bacterial_blight_20231006_164208.jpg (no label file)
    - bacterial_blight_aug_0_1092.jpg (no label file)
    - bacterial_blight_aug_0_1197.jpg (no label file)
    - bacterial_blight_aug_0_134.jpg (no label file)
    - bacterial_blight_aug_0_1362.jpg (no label file)


In [5]:
# 3. Chia train/val
random.shuffle(valid_pairs)

n_train = int(len(valid_pairs) * TRAIN_RATIO)
train_pairs = valid_pairs[:n_train]
val_pairs   = valid_pairs[n_train:]

print(f"Train : {len(train_pairs)} images")
print(f"Val   : {len(val_pairs)} images")

Train : 957 images
Val   : 240 images


In [6]:
# ── 4. Copy files ─────────────────────────────────────────────────────
def copy_pairs(pairs, split):
    img_dst = os.path.join(YOLO_DATASET, 'images', split)
    lbl_dst = os.path.join(YOLO_DATASET, 'labels', split)
    
    for img_file, label_file in pairs:
        shutil.copy(
            os.path.join(LABELING_DIR, img_file),
            os.path.join(img_dst, img_file)
        )
        shutil.copy(
            os.path.join(LABELING_DIR, label_file),
            os.path.join(lbl_dst, label_file)
        )

copy_pairs(train_pairs, 'train')
copy_pairs(val_pairs,   'val')

print(" Files copied successfully")

 Files copied successfully


In [7]:
# ── 5. Tạo data.yaml ──────────────────────────────────────────────────
dataset_abs = os.path.abspath(YOLO_DATASET)

data_yaml = {
    'path': dataset_abs,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES
}

yaml_path = os.path.join(YOLO_DATASET, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f" data.yaml saved to: {yaml_path}")
print()
with open(yaml_path) as f:
    print(f.read())

 data.yaml saved to: ../data/yolo_dataset\data.yaml

names:
- rice_leaf
nc: 1
path: D:\Python Projects\RicAI_CV_main\ric_cv\data\yolo_dataset
train: images/train
val: images/val



In [8]:
# ── 6. Kiểm tra nhanh ─────────────────────────────────────────────────
for split in ['train', 'val']:
    n_img = len(os.listdir(os.path.join(YOLO_DATASET, 'images', split)))
    n_lbl = len(os.listdir(os.path.join(YOLO_DATASET, 'labels', split)))
    match = '✅' if n_img == n_lbl else '❌'
    print(f"{match} {split:5s} → {n_img} images, {n_lbl} labels")

print("\n READY for 09_train_yolo.ipynb")

✅ train → 957 images, 957 labels
✅ val   → 240 images, 240 labels

 READY for 09_train_yolo.ipynb
